# 02 - Retrieval Experiment
Eskperimen hybrid retrieval - tuning parameter top-k, BM25 weight, dan RRF

In [1]:
import sys
from dotenv import load_dotenv

sys.path.insert(0, '../src')
load_dotenv('../.env')

True

## 1. Load Index

In [2]:
import pickle
import faiss
from core.services.rag.vector_store import load_vector_store
from core.services.rag.bm25_retriever import load_bm25
from core.services.rag.embedder import load_embedding_model

VECTOR_DIR = '../storage/vectordb'

faiss_index, chunks = load_vector_store(VECTOR_DIR)
bm25 = load_bm25(VECTOR_DIR)
model = load_embedding_model()

print(f"FAISS index loaded: {faiss_index.ntotal} vectors")
print(f"Chunks loaded: {len(chunks)}")
print(f"Embedding model: ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index loaded: 13812 vectors
Chunks loaded: 13812
Embedding model: ready


## 2. Semantic Search Only (FAISS)

In [3]:
from core.services.rag.vector_store import search_vector_store
import numpy as np

query = 'Iron requirements for 6 month old baby'
query_embedding = model.encode(query, convert_to_numpy=True)
scores, indices = search_vector_store(faiss_index, query_embedding, top_k=5)

print(f"Query: '{query}'")
print(f"\nTop 5 semantic results:")

for i, (idx, score) in enumerate(zip(indices, scores)):
    chunk = chunks[idx]
    print(f"\n[{i+1}] Score: {score:.4f}")
    print(f"Source: {chunk['metadata']['source']} p.{chunk['metadata']['page']}")
    print(f"Text: {chunk['text'][:150]}...")

Query: 'Iron requirements for 6 month old baby'

Top 5 semantic results:

[1] Score: 0.7440
Source: WHO Guideline for complementary feeding of infants and young children 6-23 months of age.pdf p.36
Text: umbilical cord clamping (82, 83). Iron 
deficiency in breastfed infants can be 
prevented more effectively by targeted 
iron supplementation than by i...

[2] Score: 0.7432
Source: Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf p.78
Text: and development, and meeting those nutrient needs can be challenging in many settings.
● Daily iron supplementation for children aged 6–23 months decr...

[3] Score: 0.7367
Source: PGS Ibu Hamil dan Ibu Menyusui - Merge-1.pdf p.42
Text: 29 
Secara rinci, kebutuhan beberapa jenis zat gizi 
mikro Ibu Hamil dapat dijelaskan sebagai berikut: 
a. Zat Besi 
Zat besi merupakan mineral yang p...

[4] Score: 0.7348
Source: Essential nutrition actions mainstreaming nutrition through the life-course (

## 3. BM25 Search Only

In [4]:
from core.services.rag.bm25_retriever import search_bm25

bm25_results = search_bm25(bm25, query, chunks, top_k=5)

print(f"Query: {query}")
print(f"\nTop 5 BM25 results:")

for i, (idx, score) in enumerate(bm25_results):
    chunk = chunks[idx]
    print(f"\n[{i+1}] Score: {score:.4f}")
    print(f"Source: {chunk['metadata']['source']} p.{chunk['metadata']['page']}")
    print(f"Text: {chunk['text'][:150]}...")

Query: Iron requirements for 6 month old baby

Top 5 BM25 results:

[1] Score: 20.5701
Source: WHO Infant and Young Child Feeding.pdf p.89
Text: needed if: 
 
K No menstruation 
K Menstruation has returned
AND 
OR
K Baby LESS than 
K Baby MORE than 
6 months old 
6 months old
AND 
OR
K Baby exc...

[2] Score: 14.5957
Source: WHO Guideline for complementary feeding of infants and young children 6-23 months of age.pdf p.44
Text: the risk of anaemia and increased Hgb 
concentrations. Children who consumed 
eggs of chickens fed with DHA-enriched 
feed also had improved DHA statu...

[3] Score: 12.8591
Source: Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf p.79
Text: requirements in the periods of rapid growth, especially in the first 5 years of life.
● Daily iron supplementation for children aged 24–59 months is a...

[4] Score: 12.1101
Source: WHO Infant and Young Child Feeding.pdf p.60
Text: 6. 
The use of the 
WHO child growt

## 4. Hybrid Search (RRF)

In [5]:
from core.services.rag.hybrid_retriever import hybrid_search

hybrid_results = hybrid_search(
    query=query,
    chunks=chunks,
    faiss_index=faiss_index,
    bm25=bm25,
    embedding_model=model,
    top_k=5
)

print(f"Query: {query}")
print(f"\nTop 5 hybrid results (RRF):")

for i, chunk in enumerate(hybrid_results):
    print(f"\n[{i+1}] RRF Score: {chunk['metadata']['rrf_score']}")
    print(f"Source: {chunk['metadata']['source']} p.{chunk['metadata']['page']}")
    print(f"Text: {chunk['text'][:150]}...")


Query: Iron requirements for 6 month old baby

Top 5 hybrid results (RRF):

[1] RRF Score: 0.016393
Source: WHO Guideline for complementary feeding of infants and young children 6-23 months of age.pdf p.36
Text: umbilical cord clamping (82, 83). Iron 
deficiency in breastfed infants can be 
prevented more effectively by targeted 
iron supplementation than by i...

[2] RRF Score: 0.016393
Source: WHO Infant and Young Child Feeding.pdf p.89
Text: needed if: 
 
K No menstruation 
K Menstruation has returned
AND 
OR
K Baby LESS than 
K Baby MORE than 
6 months old 
6 months old
AND 
OR
K Baby exc...

[3] RRF Score: 0.016129
Source: Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf p.78
Text: and development, and meeting those nutrient needs can be challenging in many settings.
● Daily iron supplementation for children aged 6–23 months decr...

[4] RRF Score: 0.016129
Source: WHO Guideline for complementary feeding of infants and yo

## 5. Reranking

In [6]:
from core.services.rag.reranker import load_reranker, rerank_chunks

reranker = load_reranker()
reranked = rerank_chunks(query, hybrid_results, reranker, top_k=3)

print(f"Query: {query}")
print(f"\nTop 3 after reranking:")

for i, chunk in enumerate(reranked):
    print(f"\n[{i+1}] Reranker Score: {chunk['metadata']['reranker_score']}")
    print(f"Source: {chunk['metadata']['source']} p.{chunk['metadata']['page']}")
    print(f"Text: {chunk['text'][:200]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Iron requirements for 6 month old baby

Top 3 after reranking:

[1] Reranker Score: 4.003911
Source: WHO Guideline for complementary feeding of infants and young children 6-23 months of age.pdf p.44
Text: the risk of anaemia and increased Hgb 
concentrations. Children who consumed 
eggs of chickens fed with DHA-enriched 
feed also had improved DHA status. The 
modelling study found that when meat, 
pou...

[2] Reranker Score: 2.660953
Source: Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf p.78
Text: and development, and meeting those nutrient needs can be challenging in many settings.
● Daily iron supplementation for children aged 6–23 months decreases the risk of anaemia, iron 
deficiency and ir...

[3] Reranker Score: -2.64489
Source: WHO Infant and Young Child Feeding.pdf p.89
Text: needed if: 
 
K No menstruation 
K Menstruation has returned
AND 
OR
K Baby LESS than 
K Baby MORE than 
6 months old 
6 months old
AN

## 6. Compare Queries

In [ ]:
from core.services.rag.reranker import rerank_chunks

test_queries = [
    'stunting preventation toddlers',
    'kapan mulai MPASI bayi',
    'kebutuhan zar besi anak 6 bulan',
    'ASI ekslusif berapa lama'
]

for q in test_queries:
    candidates = hybrid_search(q, chunks, faiss_index, bm25, model, top_k=10)
    results = rerank_chunks(q, candidates, reranker, top_k=3)
    
    print(f"Query: \"{q}\"")
    for r in results[:2]:
        print(f"  → {r['metadata']['source']} p.{r['metadata']['page']} | score: {r['metadata']['reranker_score']:.3f}")
        print(f"     {r['text'][:100]}...")
    print()

Query: "stunting preventation toddlers"
  → JME-Brochure-UNSDG-regions-2025-v23 (UNICEF Malnutrition Report).pdf p.2 | score: -1.904
     is critical to monitor and analyse 
country, regional and global progress 
going forward.
Defining t...
  → Stranas_Percepatan_Pencegahan_Anak_Kerdil.pdf p.29 | score: -6.120
     STRATEGI NASIONAL PERCEPATAN PENCEGAHAN STUNTING PERIODE 2018-2024
21
27. Berdasarkan Riskesdas 2018...

Query: "kapan mulai MPASI bayi"
  → Pedoman Gizi Seimbang — Kemenkes RI 2014.pdf p.65 | score: 6.202
     kemampuan pencernaan bayi atau anak. 
2) 
Kapan bayi mendapat MP-ASI? 
Mulai usia 6 bulan sampai den...
  → Buku KIA.pdf p.31 | score: 5.512
     58
59
Pemenuhan Gizi Usia 6 - 24 Bulan
1. Tepat waktu
MPASI diberikan saat ASI saja sudah tidak 
dap...

Query: "kebutuhan zar besi anak 6 bulan"
  → PGS Ibu Hamil dan Ibu Menyusui - Merge-1.pdf p.107 | score: 6.710
     sesuai dosis dan jumlahnya selama kehamilan 
akan dapat mencukupi kebutuhan zat besi bayi 
sampai us...
